# Address Extraction with Qwen 3.5

Clean messy address text into structured **street address, city, state, ZIP, and rationale** fields using few-shot examples and a local model running in Google Colab.

1. Select a GPU under **Runtime → Change runtime type**.
2. Run the cells from top to bottom.
3. Upload your examples CSV and your input CSV when prompted.
4. Download the cleaned results.

**Examples CSV columns:** `raw_text`, `address`, `city`, `state`, `zip`, `rationale`  
**Input CSV column:** `raw_text` (an `id` column is optional).

The number of examples and number of input rows are determined from your uploaded files. No filename or state is assumed.

## 1. Install dependencies

In [ ]:
import sys
import subprocess

packages = [
    "pandas==2.2.3",                  # Google Colab compatibility
    "transformers>=5.0,<6",
    "accelerate>=1.4,<2",
    "bitsandbytes>=0.46,<1",
    "safetensors",
    "sentencepiece",
    "einops",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *packages]
)
print("Dependencies installed. Keep Colab's existing PyTorch installation.")

Dependencies installed. Keep Colab's existing PyTorch installation.


## 2. GPU and optional optimizations

Qwen's linear-attention layers can use `causal_conv1d` and `flash-linear-attention` when compatible kernels are available. The T4 does not support every prebuilt CUDA kernel. This section reports what is available and tries **binary wheels only** to avoid unexpectedly compiling large CUDA extensions.

An unavailable optimization is a performance limitation, not a broken address extraction process.

In [ ]:
import importlib.util
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU in Runtime → Change runtime type, then rerun.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

OPTIONAL_OPTIMIZATIONS = {
    "causal_conv1d": "causal-conv1d",
    "fla": "flash-linear-attention",
}
TRY_PREBUILT_OPTIMIZATIONS = True

for module, package in OPTIONAL_OPTIMIZATIONS.items():
    if importlib.util.find_spec(module) is None and TRY_PREBUILT_OPTIMIZATIONS:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:",
             "--no-deps", package],
            capture_output=True, text=True
        )
        importlib.invalidate_caches()
        if result.returncode != 0:
            print(f"{package}: prebuilt wheel unavailable for this environment.")
    if importlib.util.find_spec(module) is None:
        print(f"{module}: using PyTorch fallback.")
    else:
        try:
            __import__(module)
            print(f"{module}: import successful (runtime GPU support not yet verified).")
        except Exception as exc:
            print(f"{module}: installed but could not import ({type(exc).__name__}).")

GPU: Tesla T4
Compute capability: (7, 5)
PyTorch: 2.11.0+cu130 | CUDA: 13.0
causal-conv1d: prebuilt wheel unavailable for this environment.
causal_conv1d: using PyTorch fallback.
fla: import successful (runtime GPU support not yet verified).


## 3. Upload example CSV

In [ ]:
from google.colab import files
import pandas as pd
import io

print("Choose your few-shot examples CSV.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select exactly one examples CSV.")
examples_filename, examples_bytes = next(iter(uploaded.items()))
examples_df = pd.read_csv(io.BytesIO(examples_bytes), dtype=str, keep_default_na=False, encoding="utf-8-sig")
required_examples = {"raw_text", "address", "city", "state", "zip", "rationale"}
missing = required_examples.difference(examples_df.columns)
if missing:
    raise ValueError(f"Examples CSV is missing columns: {sorted(missing)}")
if examples_df.empty or examples_df["raw_text"].str.strip().eq("").any():
    raise ValueError("Examples must contain nonempty raw_text values.")
print(f"Loaded {len(examples_df)} examples from {examples_filename}.")
display(examples_df.head())

Choose your few-shot examples CSV.


Saving illinois_address_few_shot_20_with_state.csv to illinois_address_few_shot_20_with_state.csv
Loaded 20 examples from illinois_address_few_shot_20_with_state.csv.


,example_id,raw_text,address,city,state,zip,rationale
0,1,"ATTN: Sam Taylor | 82 W Randolph St Ste 710, C...",82 W Randolph St,Chicago,IL,60602,Removed recipient and suite.
1,2,"c/o Jamie Reed, 1470 Oakton St Apt 3C, Des Pla...",1470 Oakton St,Des Plaines,IL,60018,Removed care-of and apartment.
2,3,PAYROLL DEPT; 412 S Neil St Floor 2; Champaign...,412 S Neil St,Champaign,IL,61820,Removed department and floor.
3,4,"Ship To: 231 E Main St, Belleville IL 62220; d...",231 E Main St,Belleville,IL,62220,Removed shipping directions and phone.
4,5,"OLD: 71 Pine St, Alton IL 62002 | CURRENT: 928...",928 Washington Ave,Alton,IL,62002,Selected current address.


## 4. Upload input CSV

In [ ]:
print("Choose the CSV containing the addresses to clean.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select exactly one input CSV.")
input_filename, input_bytes = next(iter(uploaded.items()))
input_df = pd.read_csv(io.BytesIO(input_bytes), dtype=str, keep_default_na=False, encoding="utf-8-sig")
if "raw_text" not in input_df.columns:
    raise ValueError("Input CSV must have a raw_text column.")
if input_df.empty:
    raise ValueError("Input CSV has no rows.")
if "id" not in input_df.columns:
    input_df.insert(0, "id", [str(i) for i in range(1, len(input_df) + 1)])
if not input_df["id"].is_unique:
    raise ValueError("Input id values must be unique.")
print(f"Loaded {len(input_df)} rows from {input_filename}.")
display(input_df.head())

Choose the CSV containing the addresses to clean.


Saving illinois_address_extraction_100_input.csv to illinois_address_extraction_100_input.csv
Loaded 100 rows from illinois_address_extraction_100_input.csv.


,id,raw_text
0,1,ATTN: Taylor Rivera / ACME GROUP / 153 S 5th A...
1,2,Ship to: Jordan Smith | 206 S State St | Floor...
2,3,"Accounts Payable Dept; 259 Park Blvd, Chicago ..."
3,4,c/o Casey Brooks -- 312 Oak Ave Apt 5B -- Elgi...
4,5,*** 365 RIVER DR *** BLDG 2 RM 205 *** BLOOMIN...


## 5. Load model

In [ ]:
from transformers import AutoTokenizer, Qwen3_5ForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-9B"
MAX_NEW_TOKENS = 160

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = Qwen3_5ForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()
print(f"Model loaded: {MODEL_ID}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GiB")

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3.5-9B
GPU memory allocated: 7.12 GiB


## 6. Extraction rules and examples

In [ ]:

import json
import re
import time
import torch

FIELDS = ("address", "city", "state", "zip", "rationale")

SYSTEM_PROMPT = """
Extract a physical street address for geocoding from one messy text string.

Return exactly one JSON object with these five string fields:
address, city, state, zip, rationale.

RULES:
1. Keep the primary street number, street name, and directional.
2. Extract the city, two-letter US state abbreviation, and five-digit ZIP.
3. Remove recipient names, ATTN lines, care-of information, businesses,
   departments, phone numbers, emails, invoices, reference numbers,
   websites, HTML, delivery instructions, and unnecessary punctuation.
4. Remove apartment, suite, unit, floor, room, and secondary building
   identifiers that are not needed for street-level geocoding.
5. Prefer addresses explicitly labeled CURRENT, NEW, or PHYSICAL
   over OLD addresses or mailing addresses.
6. Correct obvious typos only when supported by the original text.
7. Never invent unsupported street numbers, streets, cities, or ZIPs.
8. If no usable physical street address exists, leave address empty.
9. If multiple conflicting addresses exist without a clear choice,
   leave address empty.
10. Preserve city, state, and ZIP when recoverable, even if the
    street address cannot be determined.
11. Explain important changes or missing information in one
    short rationale.

Return valid JSON only. No markdown or additional commentary.
"""

# Build few-shot examples from the uploaded CSV.
demonstrations = []

for _, row in examples_df.iterrows():
    expected = {
        field: str(row[field])
        for field in FIELDS
    }

    demonstrations.extend([
        {
            "role": "user",
            "content": "Raw address: " + str(row["raw_text"])
        },
        {
            "role": "assistant",
            "content": json.dumps(expected, ensure_ascii=False)
        }
    ])


def parse_response(text):
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL
    ).strip()

    try:
        result = json.loads(text)

    except json.JSONDecodeError:
        match = re.search(r"\{[\s\S]*?\}", text)

        if not match:
            raise ValueError(
                "No JSON found in model response: " + text[:200]
            )

        result = json.loads(match.group())

    if not isinstance(result, dict):
        raise ValueError("Model response is not a JSON object.")

    cleaned = {
        field: str(result.get(field, "") or "").strip()
        for field in FIELDS
    }

    # Standardize state capitalization.
    if cleaned["state"]:
        cleaned["state"] = cleaned["state"].upper()

    # Convert ZIP+4 to five digits.
    if re.fullmatch(r"\d{5}-\d{4}", cleaned["zip"]):
        cleaned["zip"] = cleaned["zip"][:5]

    return cleaned


def extract_address(raw_text):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    messages.extend(demonstrations)

    messages.append({
        "role": "user",
        "content": "Raw address: " + str(raw_text)
    })

    # FIX: Return a dictionary of model inputs.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False
    )

    # Move tokenized inputs to the model device.
    inputs = inputs.to(model.device)

    # Generate the cleaned address.
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only newly generated tokens.
    input_length = inputs["input_ids"].shape[-1]

    response_text = tokenizer.decode(
        generated[0][input_length:],
        skip_special_tokens=True
    )

    return parse_response(response_text)


print(f"Ready: {len(demonstrations) // 2} example pairs loaded.")

# TEST ONE ADDRESS BEFORE PROCESSING THE DATASET.
print("\nTesting the first address...")

test_result = extract_address(input_df.iloc[0]["raw_text"])

print("\nOriginal:")
print(input_df.iloc[0]["raw_text"])

print("\nQwen extraction:")
print(json.dumps(test_result, indent=2))


Ready: 20 example pairs loaded.

Testing the first address...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.



Original:
ATTN: Taylor Rivera / ACME GROUP / 153 S 5th Ave Suite 101, Joliet, IL 60432 / PLEASE CALL

Qwen extraction:
{
  "address": "153 S 5th Ave",
  "city": "Joliet",
  "state": "IL",
  "zip": "60432",
  "rationale": "Removed recipient, business name, suite, and delivery instructions."
}


## 7. Extract addresses

Results are saved to a CSV after each row. The filename is derived from the **uploaded input filename**.

By default, all rows are processed. Set `MAX_ROWS` to a small integer if you'd like to test a subset first.

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

MAX_ROWS = None
input_stem = Path(input_filename).stem
output_path = Path("/content") / f"{input_stem}_cleaned.csv"

subset = input_df if MAX_ROWS is None else input_df.head(MAX_ROWS)
results = []

for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Cleaning"):
    start = time.perf_counter()
    try:
        extracted = extract_address(row["raw_text"])
        error = ""
    except Exception as exc:
        extracted = {field: "" for field in FIELDS}
        error = f"{type(exc).__name__}: {str(exc)[:250]}"
    results.append({
        "id": row["id"],
        "raw_text": row["raw_text"],
        **extracted,
        "error": error,
        "seconds": round(time.perf_counter() - start, 2),
    })
    pd.DataFrame(results).to_csv(output_path, index=False, encoding="utf-8-sig")

results_df = pd.DataFrame(results)
print(f"Saved {len(results_df)} rows to {output_path.name}")
print(f"Rows with processing errors: {results_df['error'].ne('').sum()}")
print(f"Average processing time: {results_df['seconds'].mean():.2f} sec/row")
display(results_df.head(10))

Cleaning:   0%|          | 0/100 [00:00<?, ?it/s]

Saved 100 rows to illinois_address_extraction_100_input_cleaned.csv
Rows with processing errors: 0
Average processing time: 7.05 sec/row


,id,raw_text,address,city,state,zip,rationale,error,seconds
0,1,ATTN: Taylor Rivera / ACME GROUP / 153 S 5th A...,153 S 5th Ave,Joliet,IL,60432,"Removed recipient, business name, suite, and d...",,7.73
1,2,Ship to: Jordan Smith | 206 S State St | Floor...,206 S State St,Urbana,IL,61801,"Removed recipient, floor, and phone.",,6.62
2,3,"Accounts Payable Dept; 259 Park Blvd, Chicago ...",259 Park Blvd,Chicago,IL,60614,Removed department label and invoice number.,,6.93
3,4,c/o Casey Brooks -- 312 Oak Ave Apt 5B -- Elgi...,312 Oak Ave,Elgin,IL,60120,Removed care-of and apartment.,,6.25
4,5,*** 365 RIVER DR *** BLDG 2 RM 205 *** BLOOMIN...,365 River Dr,Bloomington,IL,61701,Removed decorative asterisks and secondary bui...,,7.45
5,6,"PO BOX 206; PHYSICAL ADDRESS: 418 Broadway, Ch...",418 Broadway,Chicago,IL,60622,Selected physical address over PO Box.,,6.26
6,7,"OLD ADDRESS: 407 Pine St, Schaumburg IL 60193;...",471 E Market St,Schaumburg,IL,60193,Selected new address over old.,,7.08
7,8,<div>Deliver to 524 E Washington St&nbsp;Norma...,524 E Washington St,Normal,IL,61761,Removed HTML tags and customer reference.,,6.52
8,9,"Ref#387 | 577 St Charles Rd, Evanston IL 60201...",577 St Charles Rd,Evanston,IL,60201,Removed reference number and email.,,6.77
9,10,"=""630 W Jefferson St, Peoria, IL 61602"" (leave...",630 W Jefferson St,Peoria,IL,61602,"Removed spreadsheet wrapper, punctuation, and ...",,7.01


## 8. Download cleaned CSV

In [ ]:
from google.colab import files
if not output_path.exists():
    raise FileNotFoundError("Run extraction before downloading.")
files.download(str(output_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>